In [23]:
import os
import re    # regular expressions
import string    # string manipulation
from collections import Counter
import copy
import numpy as np
import scipy
import pandas
import matplotlib.pyplot as plt

# Multinomial Naive Bayes classification on text data

The goal of this exercise is to use a naive Bayes model to classify text files. Each text file is given a category ("business", "entertainment", "politics", "sport", "tech"), and the goal is to infer this category from an analysis of the text.

In [64]:
path_dataset = "data/bbc/bbc"
path_stopwords = "data/bbc/bbc/stopwords-en.txt"
list_classes = ["business", "entertainment", "politics", "sport", "tech"]
n_classes = len(list_classes)

**Preliminary question**

Open some of the text files and take a look at the data.

## Load the data

First, the data must be loaded. One must load the "stop-words", that are the words that are commonly used in English, and are not believed to contain any crucial information about the theme of the text. Then, the files must be loaded.

**Question 1**

Store the content of the stopwords file into a set of strings. Add to it the empty string "".

One can use the string method `rstrip` to remove the trailing blank characters.

In [60]:
def load_stopwords(path_stopwords):
    return [s.rstrip() for s in open(path_stopwords)]

In [61]:
set_stopwords = set(load_stopwords(path_stopwords) + [""])

**Question 2**

Write a function loading the dataset into a dictionary `dct_data`:
 * whose keys are the classes names;
 * whose values are the lists of the contents of the text files.
For instance, `dct_data["business"]` returns a list of strings, where each string is the content of a file (in the folder "business").

One can use the function `os.path.join` to build paths to files, `os.listdir` to list all the files contained in a directory and the file method `read` to gather the entire content of a file.

In [67]:
def load_dataset(path_dataset, list_classes):
    dct_data = {}
    for c in list_classes:
        lst_data = []
        
        # List of all text files in the folder of the current class
        lst_files = os.listdir(os.path.join(path_dataset, c))
        for curr_file in lst_files:
            with open(os.path.join(path_dataset, c, curr_file), encoding = "latin-1") as f:
                lst_data.append(f.read())
        dct_data[c] = lst_data
    return dct_data

In [68]:
dct_data = load_dataset(path_dataset, list_classes)

## Preprocess the data

We want to use perform Multinomial Naive Bayes on this dataset. To use it, we have first to precprocess the data (clean the text, prepare the data for the naive Bayes method).

**Question 3**

Write a function to remove the punctuation from a string, a function to transform all characters of a string into lower-case characters, and a function that transforms a string into the list of words it contains.

One can use the string method `translate`, the static method `str.maketrans`, the in-built string `string.punctuation`, and `re.split`.

In [69]:
def remove_punctuation(s):
    return s.translate(str.maketrans('', '', string.punctuation))

def lower_case(s):
    return s.lower()

def split_into_words(text):
    return re.split(r"\W+", text)

**Question 4**

Process the texts to transform them into lists of words.

In [75]:
dct_data_words = {k: [split_into_words(lower_case(remove_punctuation(s))) for s in v] for k, v in dct_data.items()}

**Question 5**

Create the set of all the different words present in the texts, excuding the stopwords. 

In [78]:
set_words = set()
for k, v in dct_data_words.items():
    for text in v:
        for s in text:
            if not s in set_stopwords:
                set_words.add(s)
                

**Question 6**

Create the array of features of the dataset. This array must be of size n * p, where n is the number of texts, and p is the number of different words in the dataset. Each row i must contain the number of occurrences of each word. For example, the coefficient at (i, j) contains the number of occurrences of the word n° j in the text n° i. Create also the array of targets: this should be an array of size n, where the coefficient n° i contains a number representing the class of the text n° i.

Note: one should give one number to each word of the set and to each class.

The object `collections.Counter` can be very useful in this situation.

In [73]:
dct_word_labels = {}
for w in set_words:
    dct_word_labels[w] = len(dct_word_labels)
    
dct_classes_labels = {}
for cl in list_classes:
    dct_classes_labels[cl] = len(dct_classes_labels)

In [79]:
n = sum(len(v) for k, v in dct_data_words.items()) ##number of texts
p = len(set_words) ##number of different words in the data set 

X = np.zeros((n, p)) ## it is the vector of all 0 of n*p
y = np.zeros(n) ## it is the target that we will fill in 

idx = 0
for k, v in dct_data_words.items():
    for text in v:
        counter = Counter(text)
        for word, count in counter.items():
            if word in set_words:
                X[idx, dct_word_labels[word]] = count ## X(i,j) = count
        y[idx] = dct_classes_labels[k] ## the clas
        idx += 1

## By-hand multinomial naive Bayes

**Question 7**

Split the dataset into a training set and a test set (we do not need a validation set since we do not perform model selection).

The first step of naive Bayes is to find the distribution that fits the best to the training data. Given a class $c$, with the multinomial distribution $\mathcal{M}(\theta_1^c, \theta_2^c, \cdots, \theta_p^c, p)$, we want to find the parameters $\hat{\theta}_j^c$ maximizing the likehood of the training data. We have:
$$
\hat{\theta}_j^c = \frac{n_j^c}{\sum_{k = 1}^p n_k^c} ,
$$
where $n_j^c$ is the number of occurrences of the word n° j in the texts with label c.

We will not use this formula, but a regularized version of it:
$$
\hat{\theta}_j^c = \frac{n_j^c + \alpha}{\sum_{k = 1}^p n_k^c + \alpha p} ,
$$
where $\alpha > 0$ to avoid a zero denominator. We can set $\alpha = 1$.

Compute the $\hat{\theta}_j^c$ for each class c and each word j.

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2)

NameError: name 'X' is not defined

In [35]:
# We build an array in which we count the number of occurrences of each word in the set
# of all texts in each class
n_occurrences = np.zeros((n_classes, p))

for x, c in zip(X_train, y_train):
    n_occurrences[int(c)] += x

NameError: name 'p' is not defined

In [36]:
alpha = 1.
theta = ((n_occurrences.T + alpha) / (n_occurrences.sum(1) + alpha * p)).T

NameError: name 'n_occurrences' is not defined

**Question 8**

Predict the classes of the test dataset. From a Bayesian point of view, for each data point, the chosen class should maximize the posterior distribution.

In [37]:
# Note: the constant part has been removed from the posterior distribution

def predict(theta, X_test):
    y_hat = np.zeros(X_test.shape[0])
    for i, x in enumerate(X_test):
        lst_log_posteriors = []
        for c in range(n_classes):
            lst_log_posteriors.append((x * np.log(theta[c])).sum())
        y_hat[i] = np.argmax(lst_log_posteriors)
    return y_hat

In [38]:
y_hat = predict(theta, X_test)

NameError: name 'theta' is not defined

In [39]:
(y_hat == y_test).mean()

NameError: name 'y_hat' is not defined

## Multinomial naive Bayes with sklearn

**Question 9**

Split the dataset into a training set and a test set (we do not need a validation set since we do not perform model selection).

Use `sklearn.naive_bayes.MultinomialNB`: train it on the train set and compute the score on the test set. What do the score represent? Comment the efficiency of naive Bayes in this situation.

In [40]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2)

In [41]:
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

mnb.score(X_test, y_test)

NameError: name 'X_train' is not defined

The score represents the proportion of correct prediction. It is close to 1, so naive Bayes performs well on this dataset.

In [42]:
(mnb.predict(X_test) == y_test).mean()

NameError: name 'X_test' is not defined

# Gaussian naive Bayes

We will perform Gaussian naive Bayes on the "iris" dataset of sklearn. There are 4 features and 3 classes.

## Load the dataset

**Question 1**

Load the "Iris" dataset. and split it into a training set and a test set.

In [ ]:
from sklearn.datasets import load_iris

iris_data = load_iris()
X = iris_data.data
y = iris_data.target

n, p = X.shape
n_classes = len(iris_data.target_names)

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2)

NameError: name 'X' is not defined

## By-hand Gaussian naive Bayes

**Question 2**

For each class, for each feature, compute the optimal parameters of the Gaussian (used to model the current feature).

In [44]:
_, counts = np.unique(y_train, return_counts = True) ##the unique value and count of each

##the optimal parameters of the Gaussian
mu = np.zeros((n_classes, p)) ## mean
sigma2 = np.zeros((n_classes, p)) ##variances
 
for i, x in enumerate(X_train):
    c = y_train[i]
    mu[c] += x / counts[c]

for i, x in enumerate(X_train):
    c = y_train[i]
    sigma2[c] += (x - mu[c])**2 / counts[c]

NameError: name 'y_train' is not defined

**Question 3**

Compute the prior over the classes based on the population inside each class.

Predict the class of the data points belonging to the test set.

In [45]:
priors = counts / counts.sum() 

NameError: name 'counts' is not defined

In [46]:
# Note: the constant part has been removed from the posterior distribution

def predict(mu, sigma2, X_test):
    y_hat = np.zeros(X_test.shape[0])
    for i, x in enumerate(X_test):
        lst_log_posteriors = []
        for c in range(n_classes):
            log_post = - (x - mu[c])**2 / (2 * sigma2[c]) - .5 * np.log(sigma2[c]) + np.log(priors[c])
            lst_log_posteriors.append(log_post.sum())
        y_hat[i] = np.argmax(lst_log_posteriors)
    return y_hat

In [47]:
y_hat = predict(mu, sigma2, X_test)

NameError: name 'mu' is not defined

In [48]:
(y_hat == y_test).mean()

NameError: name 'y_hat' is not defined

## Gaussian naive Bayes with sklearn

**Question 4**

Perform the Gaussian naive Bayes with sklearn.

In [49]:
from sklearn.naive_bayes import GaussianNB

In [50]:
mnb = GaussianNB()
mnb.fit(X_train, y_train)

mnb.score(X_test, y_test)

NameError: name 'X_train' is not defined

In [51]:
(mnb.predict(X_test) == y_test).mean()

NameError: name 'X_test' is not defined

**Question 5**

Compare with the knn classifier with $k = 5$ neighbors by using cross-validation.

In [52]:
from sklearn.model_selection import cross_val_score

n_splits = 15
lst_scores = cross_val_score(mnb, X_train, y_train, cv = n_splits)
print(f"Average valid score: {sum(lst_scores)/n_splits:.3f}")

NameError: name 'X_train' is not defined

In [53]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors = 5, metric = "l2")
lst_scores = cross_val_score(knn, X_train, y_train, cv = n_splits)
print(f"Average valid score: {sum(lst_scores)/n_splits:.3f}")

NameError: name 'X_train' is not defined

# Learning from multimodal features

We will process a dataset of predictive maintenance. The goal is to predict the failure type on a machine in function of physical measures and the type of machine. So, we are in a classification setup with multimodal features (categorical features and numerical features).

In [2]:
df = pandas.read_csv("predictive_maintenance.csv")
features = ['Type',
 'Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]',
 'Target',
 'Failure Type']
df

NameError: name 'pandas' is not defined

**Question 1**

Select the features `features` in the dataset.

Transform the strings in the columns `Type` and `Failure Type` by integers (which will represent the classes). One can use the method `unique` of `Series` and the method `replace_strict` of `Expr`.

Split the columns of the dataset in 3:
1. categorical features;
2. numerical features;
3. targets.

In [1]:
df = df[features]

dct_modality = {k: v for v, k in enumerate(list(df["Type"].unique()))}
dct_fail = {k: v for v, k in enumerate(list(df["Failure Type"].unique()))}

df = df.replace({"Type": dct_modality, "Failure Type": dct_fail})

features = ['Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]']

X_cat = df[["Type"]].to_numpy(dtype = int)
X_num = df[features].to_numpy()
y = df["Failure Type"].to_numpy(dtype = int)

NameError: name 'df' is not defined

**Question 2**

Split the dataset into a train set and a test set.

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, CategoricalNB
from sklearn.model_selection import train_test_split

Xc_train, Xc_test, Xn_train, Xn_test, y_train, y_test = train_test_split(X_cat, X_num, y, test_size = .2)

**Question 3**

Perform naive Bayes on the categorial features and on the numerical features.

In [ ]:
cnb = CategoricalNB()
cnb.fit(Xc_train, y_train)

cnb.score(Xc_test, y_test)

In [ ]:
gnb = GaussianNB()
gnb.fit(Xn_train, y_train)

gnb.score(Xn_test, y_test)

**Question 4**

Combine the results and compute the accuracy when taking into account both the numerical features and the categorical features.

One can access the results of the preceding naive Bayes computations by looking at the attributes and methods of `CategoricalNB` and `GaussianNB` (`class_prior_`, `predict_log_proba`, etc.).

Is the combined result better than the two individual results? Why?

In [ ]:
# We add:
#    * the log-prior;
#    * the log-likelihoods given by the numerical features (log-posterior - log-prior);
#    * the log-likelihoods given by the categorical features (log-posterior - log-prior).
log_probs = cnb.class_log_prior_ \
    + gnb.predict_log_proba(Xn_test) - np.log(gnb.class_prior_) \
    + cnb.predict_log_proba(Xc_test) - cnb.class_log_prior_
score = (np.argmax(log_probs, axis = 1) == y_test).mean()
print(f"score: {score}")

Apparently, the same amount of information is contained in the two subsets of features, so combining them does not help.